# Day 19 — K线形态 + 指标背离

**目标**：识别关键K线形态，理解顶背离/底背离是技术分析最强的反转信号

- 🌅 上午：K线形态——锤子线、吞没、十字星、早晨之星
- ☀️ 下午：指标背离——价格与RSI/MACD的「分歧」
- 🌙 晚上：背离实战回测 + 三股扫描

In [ ]:
# ===== 基础设置 =====
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager

font_manager.fontManager.addfont('/System/Library/Fonts/Hiragino Sans GB.ttc')
plt.rcParams['font.family'] = 'Hiragino Sans GB'
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪')

## 🌅 上午：K线形态识别

K线包含四个价格：开盘、收盘、最高、最低。单根K线的形状本身就传递了大量信息。

### 实体与影线

```
  最高 ─── 上影线
  开盘/收盘 ─── 实体（阳线：收盘>开盘，红色；阴线：收盘<开盘，绿色）
  收盘/开盘 ─── 实体
  最低 ─── 下影线
```

今天要识别四种经典形态：

| 形态 | 特征 | 含义 |
|------|------|------|
| 锤子线 | 下影线≥2倍实体，上影线很短 | 跌不下去，有人抄底 |
| 看涨吞没 | 今日阳线实体完全包住昨日阴线实体 | 多头压倒空头 |
| 十字星 | 开盘≈收盘，实体极小 | 多空平衡，即将变盘 |
| 早晨之星 | 阴线→十字星→阳线，三根K线组合 | 底部反转信号 |

In [ ]:
# 加载数据
df = pd.read_csv('../data/茅台.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index()

print(f'数据: {len(df)} 行, {df.index[0].date()} ~ {df.index[-1].date()}')

In [ ]:
# ===== 计算K线基础属性 =====

# 实体（绝对值）
df['实体'] = abs(df['收盘'] - df['开盘'])

# 上影线
df['上影'] = df['最高'] - df[['开盘','收盘']].max(axis=1)

# 下影线
df['下影'] = df[['开盘','收盘']].min(axis=1) - df['最低']

# 阴阳
df['阳线'] = df['收盘'] > df['开盘']
df['阴线'] = df['收盘'] < df['开盘']

# 前一日
df['前收'] = df['收盘'].shift(1)
df['前开'] = df['开盘'].shift(1)
df['前阳'] = df['阳线'].shift(1)
df['前阴'] = df['阴线'].shift(1)

print('K线属性计算完成')
print(f'  阳线天数: {df["阳线"].sum()}  阴线天数: {df["阴线"].sum()}')

In [ ]:
# ===== 形态1: 锤子线 (Hammer) =====
# 条件: 下影线 >= 2倍实体, 上影线 <= 0.3倍实体, 出现在下跌中

df['锤子'] = (
    (df['下影'] >= 2 * df['实体']) &           # 下影线至少是实体的2倍
    (df['上影'] <= 0.3 * df['实体']) &          # 上影线很短
    (df['实体'] > 0) &                           # 有实体（不是一字板）
    (df['收盘'] < df['收盘'].shift(5))           # 前5天整体下跌
)

hammers = df[df['锤子']]
print(f'锤子线出现次数: {len(hammers)}')
if len(hammers) > 0:
    print(f'\n最近5次锤子线:')
    for date, row in hammers.tail(5).iterrows():
        print(f'  {date.date()}  开{row["开盘"]:.1f} 收{row["收盘"]:.1f}  '  
              f'下影={row["下影"]:.1f} 实体={row["实体"]:.1f}')

In [ ]:
# ===== 形态2: 看涨吞没 (Bullish Engulfing) =====
# 条件: 今日阳线, 昨日阴线, 今日实体完全包住昨日实体

df['吞没'] = (
    df['阳线'] &                                 # 今天是阳线
    df['前阴'] &                                 # 昨天是阴线
    (df['开盘'] < df['前收']) &                   # 今日开盘低于昨日收盘
    (df['收盘'] > df['前开'])                     # 今日收盘高于昨日开盘
)

engulfing = df[df['吞没']]
print(f'看涨吞没出现次数: {len(engulfing)}')
if len(engulfing) > 0:
    print(f'\n最近5次:')
    for date, row in engulfing.tail(5).iterrows():
        prev = df.iloc[df.index.get_loc(date)-1]
        print(f'  {date.date()}  昨阴({prev["开盘"]:.1f}→{prev["收盘"]:.1f}) '  
              f'今阳({row["开盘"]:.1f}→{row["收盘"]:.1f}) ✓吞没')

In [ ]:
# ===== 形态3: 十字星 (Doji) =====
# 条件: 实体 <= 总长度(最高-最低)的10%, 即开盘和收盘几乎一样

df['K线长度'] = df['最高'] - df['最低']
df['十字星'] = (df['实体'] <= 0.1 * df['K线长度']) & (df['K线长度'] > 0)

dojis = df[df['十字星']]
print(f'十字星出现次数: {len(dojis)} ({len(dojis)/len(df)*100:.1f}%)')
if len(dojis) > 0:
    print(f'\n最近5次:')
    for date, row in dojis.tail(5).iterrows():
        print(f'  {date.date()}  开{row["开盘"]:.1f} 收{row["收盘"]:.1f}  实体仅{row["实体"]:.1f}元')

In [ ]:
# ===== 形态4: 早晨之星 (Morning Star) =====
# 三日组合: 第一天阴线 → 第二天十字星(或小实体) → 第三天阳线
# 且第三天收盘超过第一天实体的一半

df['前2阴'] = df['阴线'].shift(2)
df['前1十字'] = df['十字星'].shift(1)
df['前2开'] = df['开盘'].shift(2)
df['前2收'] = df['收盘'].shift(2)

df['早晨之星'] = (
    df['前2阴'] &                               # 两天前是大阴线
    df['前1十字'] &                              # 一天前是十字星（犹豫）
    df['阳线'] &                                 # 今天是阳线（确认反转）
    (df['收盘'] > (df['前2开'] + df['前2收']) / 2)  # 今天收盘超过前阴线中点
)

morning_stars = df[df['早晨之星']]
print(f'早晨之星出现次数: {len(morning_stars)}')
if len(morning_stars) > 0:
    print(f'\n最近5次:')
    for date, row in morning_stars.tail(5).iterrows():
        d1 = df.iloc[df.index.get_loc(date)-2]
        d2 = df.iloc[df.index.get_loc(date)-1]
        print(f'  {date.date()}  D-2阴({d1["开盘"]:.1f}→{d1["收盘"]:.1f}) '  
              f'D-1星({d2["开盘"]:.1f}→{d2["收盘"]:.1f}) '  
              f'D阳({row["开盘"]:.1f}→{row["收盘"]:.1f})')

In [ ]:
# ===== 形态回测：出现后5天表现 =====

def backtest_5d(signals, name):
    """回测信号出现后5天的涨跌表现"""
    results = []
    for date in df[signals].index:
        pos = df.index.get_loc(date)
        if pos + 5 < len(df):
            ret = (df['收盘'].iloc[pos+5] / df['收盘'].iloc[pos] - 1) * 100
            results.append(ret)
    if results:
        win_rate = sum(r > 0 for r in results) / len(results) * 100
        avg_ret = np.mean(results)
        print(f'{name}: {len(results)}次 | 胜率 {win_rate:.0f}% | 均值 {avg_ret:.2f}%')
    else:
        print(f'{name}: 无足够数据')
    return results

print('K线形态 5日后表现回测:')
print('=' * 45)
backtest_5d('锤子', '锤子线    ')
backtest_5d('吞没', '看涨吞没  ')
backtest_5d('十字星', '十字星    ')
backtest_5d('早晨之星', '早晨之星  ')

### K线形态小结

单根K线不可靠——任何形态都要结合趋势和成交量判断。
- 下跌趋势末端的锤子线 > 震荡市中的锤子线
- 放量的吞没形态 > 缩量的吞没形态
- 十字星只是"可能变盘"，不是"一定变盘"

> 真正的高手不看单根K线，看的是**价格和指标的背离**——这是下午的内容。

## ☀️ 下午：指标背离——技术分析的核心

### 什么是背离？

背离（Divergence）= **价格方向和指标方向不一致**

```
顶背离 = 价格创新高，但指标没创新高（甚至更低）
       → 上涨动力在衰减，即将下跌
       
底背离 = 价格创新低，但指标没创新低（甚至更高）
       → 下跌动力在衰减，即将上涨
```

直觉：就好像一个运动员（价格）在冲刺，但心跳（指标）反而慢下来了——说明他在硬撑，马上要倒了。

### 可以用哪些指标来检测背离？

- **RSI 背离**：最常用，反应灵敏
- **MACD 背离**：更稳健，假信号少
- **成交量背离**：价格涨但量缩 = 虚涨

In [ ]:
# ===== 计算 RSI(14) =====

def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI'] = calc_rsi(df['收盘'], 14)

# 验证：和 Day17 结果一致
print(f'当前 RSI(14) = {df["RSI"].iloc[-1]:.1f}')

In [ ]:
# ===== 计算 MACD =====

ema12 = df['收盘'].ewm(span=12, adjust=False).mean()
ema26 = df['收盘'].ewm(span=26, adjust=False).mean()
df['DIF'] = ema12 - ema26
df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
df['MACD柱'] = 2 * (df['DIF'] - df['DEA'])

print(f'当前 DIF={df["DIF"].iloc[-1]:.2f}  DEA={df["DEA"].iloc[-1]:.2f}')

In [ ]:
# ===== 找局部极值点（用于检测背离） =====
# 用 scipy 的 argrelextrema，但为保持零依赖，自己实现简单的峰谷检测

def find_peaks(series, order=5):
    """找局部高点：比前后 order 个点都高"""
    peaks = []
    for i in range(order, len(series)-order):
        val = series.iloc[i]
        if all(val > series.iloc[i-order:i]) and all(val > series.iloc[i+1:i+order+1]):
            peaks.append(i)
    return peaks

def find_valleys(series, order=5):
    """找局部低点：比前后 order 个点都低"""
    valleys = []
    for i in range(order, len(series)-order):
        val = series.iloc[i]
        if all(val < series.iloc[i-order:i]) and all(val < series.iloc[i+1:i+order+1]):
            valleys.append(i)
    return valleys

print('极值点检测函数就绪')

In [ ]:
# ===== 检测RSI背离 =====
# 只看最近120天

d = df.iloc[-120:]
price_peaks = find_peaks(d['收盘'], order=5)
price_valleys = find_valleys(d['收盘'], order=5)
rsi_peaks = find_peaks(d['RSI'], order=5)
rsi_valleys = find_valleys(d['RSI'], order=5)

divergences = []

# 顶背离: 价格第二个峰更高，但RSI第二个峰更低
for i in range(len(price_peaks)-1):
    for j in range(i+1, len(price_peaks)):
        p1, p2 = price_peaks[i], price_peaks[j]
        # 价格创新高
        if d['收盘'].iloc[p2] > d['收盘'].iloc[p1]:
            # 找这两个峰附近的RSI峰
            rsi_near = [r for r in rsi_peaks if abs(r-p2) <= 10]
            if rsi_near:
                r2 = rsi_near[0]
                rsi_near1 = [r for r in rsi_peaks if abs(r-p1) <= 10]
                if rsi_near1:
                    r1 = rsi_near1[0]
                    if d['RSI'].iloc[r2] < d['RSI'].iloc[r1]:  # RSI没创新高
                        if abs(r2-p2) <= 5:  # RSI峰和价格峰要对齐
                            divergences.append({
                                'type': '顶背离',
                                'date': d.index[p2],
                                'price1': d['收盘'].iloc[p1],
                                'price2': d['收盘'].iloc[p2],
                                'rsi1': d['RSI'].iloc[r1],
                                'rsi2': d['RSI'].iloc[r2],
                            })
                            break

# 底背离: 价格第二个谷更低，但RSI第二个谷更高
for i in range(len(price_valleys)-1):
    for j in range(i+1, len(price_valleys)):
        v1, v2 = price_valleys[i], price_valleys[j]
        if d['收盘'].iloc[v2] < d['收盘'].iloc[v1]:
            rsi_near = [r for r in rsi_valleys if abs(r-v2) <= 10]
            if rsi_near:
                r2 = rsi_near[0]
                rsi_near1 = [r for r in rsi_valleys if abs(r-v1) <= 10]
                if rsi_near1:
                    r1 = rsi_near1[0]
                    if d['RSI'].iloc[r2] > d['RSI'].iloc[r1]:
                        if abs(r2-v2) <= 5:
                            divergences.append({
                                'type': '底背离',
                                'date': d.index[v2],
                                'price1': d['收盘'].iloc[v1],
                                'price2': d['收盘'].iloc[v2],
                                'rsi1': d['RSI'].iloc[r1],
                                'rsi2': d['RSI'].iloc[r2],
                            })
                            break

print(f'RSI背离检测结果 (近120天):')
print(f'  顶背离: {sum(1 for div in divergences if div["type"]=="顶背离")} 次')
print(f'  底背离: {sum(1 for div in divergences if div["type"]=="底背离")} 次')
for div in divergences:
    print(f'\n  {div["type"]} @ {div["date"].date()}')
    print(f'    价格: {div["price1"]:.1f}→{div["price2"]:.1f} '  
          f'({"涨" if div["price2"]>div["price1"] else "跌"}{abs(div["price2"]-div["price1"]):.1f})')
    print(f'    RSI:  {div["rsi1"]:.1f}→{div["rsi2"]:.1f} '  
          f'({"涨" if div["rsi2"]>div["rsi1"] else "跌"}{abs(div["rsi2"]-div["rsi1"]):.1f})')

In [ ]:
# ===== 背离回测：背离出现后10天的表现 =====

if divergences:
    print('背离信号回测（出现后 5/10/20 天）:')
    print('=' * 55)
    for div in divergences:
        pos = df.index.get_loc(div['date'])
        date_str = div['date'].date()
        print(f'\n{div["type"]} @ {date_str}')
        for horizon, label in [(5, '5日'), (10, '10日'), (20, '20日')]:
            if pos + horizon < len(df):
                ret = (df['收盘'].iloc[pos+horizon] / df['收盘'].iloc[pos] - 1) * 100
                direction = '✓正确' if (div['type']=='顶背离' and ret<0) or (div['type']=='底背离' and ret>0) else '✗失败'
                print(f'  {label}后: {ret:+.2f}% {direction}')
else:
    print('近120天未检测到背离信号')

In [ ]:
# ===== 可视化：背离位置标记 =====

d = df.iloc[-120:]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8),
                                gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

# 图1: 价格
ax1.plot(d.index, d['收盘'], color='#2c3e50', lw=1.5, label='收盘价')
ax1.set_title('茅台 RSI背离检测（近120天）', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格 (元)')

# 标记背离点
for div in divergences:
    color = '#e74c3c' if div['type'] == '顶背离' else '#2ecc71'
    marker = 'v' if div['type'] == '顶背离' else '^'
    label = div['type']
    ax1.scatter(div['date'], d.loc[div['date'], '收盘'], color=color, s=120,
                marker=marker, edgecolor='white', lw=1, zorder=10, label=label)

# 去重图例
handles, labels = ax1.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc='upper left', frameon=False, fontsize=10)

# 图2: RSI
ax2.plot(d.index, d['RSI'], color='#8e44ad', lw=1.2, label='RSI(14)')
ax2.axhline(70, color='#e74c3c', lw=0.8, ls='--', alpha=0.4)
ax2.axhline(30, color='#2ecc71', lw=0.8, ls='--', alpha=0.4)
ax2.axhline(50, color='#7f8c8d', lw=0.5, alpha=0.3)
ax2.fill_between(d.index, 70, 100, alpha=0.05, color='#e74c3c')
ax2.fill_between(d.index, 0, 30, alpha=0.05, color='#2ecc71')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)

# 标记背离点的RSI
for div in divergences:
    color = '#e74c3c' if div['type'] == '顶背离' else '#2ecc71'
    marker = 'v' if div['type'] == '顶背离' else '^'
    ax2.scatter(div['date'], d.loc[div['date'], 'RSI'], color=color, s=100,
                marker=marker, edgecolor='white', lw=1, zorder=10)

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 🌙 晚上：背离实战 + 多股扫描

背离的三个层次（弱→强）：
1. **单一指标背离**：仅 RSI 背离 — 信号较弱，可能是假背离
2. **多指标共振背离**：RSI + MACD 同时背离 — 信号强很多
3. **背离 + K线形态**：底背离 + 锤子线/吞没形态 — 最强的反转信号组合

背离不是一出现就反转——大趋势中背离可以持续好几周。正确的做法：
- 背离出现 → 进入观察模式（不要立刻买入）
- 等确认信号（放量突破关键均线、K线形态确认）→ 再动手

In [ ]:
# ===== 三股K线形态扫描 =====

stocks = [('茅台', '../data/茅台.csv'), ('招商银行', '../data/招商银行.csv'),
          ('比亚迪', '../data/比亚迪.csv')]

print('三只股票 K线形态 + RSI背离 扫描结果:')
print('=' * 60)

for name, path in stocks:
    s = pd.read_csv(path)
    s['日期'] = pd.to_datetime(s['日期'])
    s = s.set_index('日期').sort_index()
    
    # K线属性
    s['实体'] = abs(s['收盘'] - s['开盘'])
    s['上影'] = s['最高'] - s[['开盘','收盘']].max(axis=1)
    s['下影'] = s[['开盘','收盘']].min(axis=1) - s['最低']
    s['K线长度'] = s['最高'] - s['最低']
    s['阳线'] = s['收盘'] > s['开盘']
    s['阴线'] = s['收盘'] < s['开盘']
    
    # RSI
    s['RSI'] = calc_rsi(s['收盘'], 14)
    
    # 形态检测（只统计近120天）
    d = s.iloc[-120:]
    hammers = sum((d['下影'] >= 2 * d['实体']) & (d['上影'] <= 0.3 * d['实体']) & (d['实体'] > 0))
    dojis = sum((d['实体'] <= 0.1 * d['K线长度']) & (d['K线长度'] > 0))
    engulfing = sum(d['阳线'] & d['阴线'].shift(1).fillna(False) &
                    (d['开盘'] < d['收盘'].shift(1)) & (d['收盘'] > d['开盘'].shift(1)))
    
    # 最近背离
    d_peaks = find_peaks(d['收盘'], 5)
    d_rsi_peaks = find_peaks(d['RSI'], 5)
    d_valleys = find_valleys(d['收盘'], 5)
    d_rsi_valleys = find_valleys(d['RSI'], 5)
    
    has_top_div = False
    has_bot_div = False
    for i in range(len(d_peaks)-1):
        for j in range(i+1, min(i+3, len(d_peaks))):
            p1, p2 = d_peaks[i], d_peaks[j]
            if d['收盘'].iloc[p2] > d['收盘'].iloc[p1]:
                nr = [r for r in d_rsi_peaks if abs(r-p2)<=5]
                nr1 = [r for r in d_rsi_peaks if abs(r-p1)<=5]
                if nr and nr1 and d['RSI'].iloc[nr[0]] < d['RSI'].iloc[nr1[0]]:
                    has_top_div = True
                    break
        if has_top_div: break
    
    for i in range(len(d_valleys)-1):
        for j in range(i+1, min(i+3, len(d_valleys))):
            v1, v2 = d_valleys[i], d_valleys[j]
            if d['收盘'].iloc[v2] < d['收盘'].iloc[v1]:
                nr = [r for r in d_rsi_valleys if abs(r-v2)<=5]
                nr1 = [r for r in d_rsi_valleys if abs(r-v1)<=5]
                if nr and nr1 and d['RSI'].iloc[nr[0]] > d['RSI'].iloc[nr1[0]]:
                    has_bot_div = True
                    break
        if has_bot_div: break
    
    print(f'\n{name}:')
    print(f'  最新收盘: {s["收盘"].iloc[-1]:.2f}  |  RSI(14): {s["RSI"].iloc[-1]:.1f}')
    print(f'  锤子线: {hammers}次  |  十字星: {dojis}次  |  吞没: {engulfing}次')
    print(f'  顶背离: {"⚠️有" if has_top_div else "无"}  |  底背离: {"✅有" if has_bot_div else "无"}')

In [ ]:
# ===== MACD柱背离辅助验证 =====
# MACD柱的背离往往比RSI背离更可靠

d = df.iloc[-120:]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10),
                                gridspec_kw={'height_ratios': [2.5, 1.5, 1]}, sharex=True)

# 图1: 价格
ax1.plot(d.index, d['收盘'], color='#2c3e50', lw=1.5, label='收盘价')
ax1.set_title('茅台 价格 + RSI + MACD 综合背离视图', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格')
for div in divergences:
    color = '#e74c3c' if div['type'] == '顶背离' else '#2ecc71'
    marker = 'v' if div['type'] == '顶背离' else '^'
    ax1.scatter(div['date'], d.loc[div['date'], '收盘'], color=color, s=120,
                marker=marker, edgecolor='white', lw=1, zorder=10, label=div['type'])
handles, labels = ax1.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc='upper left', frameon=False, fontsize=9)

# 图2: RSI
ax2.plot(d.index, d['RSI'], color='#8e44ad', lw=1.2)
ax2.axhline(70, color='#e74c3c', lw=0.6, ls='--', alpha=0.4)
ax2.axhline(30, color='#2ecc71', lw=0.6, ls='--', alpha=0.4)
ax2.fill_between(d.index, 70, 100, alpha=0.05, color='#e74c3c')
ax2.fill_between(d.index, 0, 30, alpha=0.05, color='#2ecc71')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)

# 图3: MACD柱
colors = ['#e74c3c' if v >= 0 else '#2ecc71' for v in d['MACD柱']]
ax3.bar(range(len(d)), d['MACD柱'], color=colors, width=1)
ax3.plot(range(len(d)), d['DIF'], color='#2c3e50', lw=1, label='DIF')
ax3.plot(range(len(d)), d['DEA'], color='#e67e22', lw=0.8, label='DEA')
ax3.axhline(0, color='#7f8c8d', lw=0.5)
ax3.set_ylabel('MACD')
ax3.legend(loc='upper left', frameon=False, fontsize=9)

for ax in [ax1, ax2, ax3]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 📝 今日三问

把下面三个问题的答案写进 `day19/学习笔记.txt`：

1. 茅台近120天出现了哪些K线形态？各自的胜率如何？
2. 茅台是否出现RSI背离？是顶背离还是底背离？背离信号之后价格怎么走？
3. 背离为什么比单一指标（如只看RSI超买超卖）更可靠？什么情况下背离会失效？